In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import cv2
import random
from mtrain.utils import show, mkdir, show_single_channel_red_green_black, rd_bk_gn
from mtrain.disk import DiskImage, DiskBooleanMask

In [ ]:
def single_conv_operation(patch, kernel):
    """
    Performs a single convolution operation at one point.

    Args:
        patch: 2D numpy array representing the image patch (same size as kernel)
        kernel: 2D numpy array representing the convolution kernel

    Returns:
        Single scalar value - the result of element-wise multiplication and sum
    """
    return np.sum(patch * kernel)


def conv2d_pytorch_style(image, kernel):
    # Get dimensions
    img_h, img_w = image.shape
    kernel_h, kernel_w = kernel.shape

    # Calculate output dimensions (valid convolution)
    output_h = img_h - kernel_h + 1
    output_w = img_w - kernel_w + 1

    # Initialize output array
    output = np.zeros((output_h, output_w))

    # Slide kernel across the image
    for i in range(output_h):
        for j in range(output_w):
            # Extract patch from image
            patch = image[i : i + kernel_h, j : j + kernel_w]
            # Apply single convolution operation
            output[i, j] = single_conv_operation(patch, kernel)

    return output


def get_patch_and_kernel_center(image, kernel):
    # Get dimensions
    img_h, img_w = image.shape
    kernel_h, kernel_w = kernel.shape

    # Calculate kernel center (radius)
    center_h = kernel_h // 2
    center_w = kernel_w // 2

    # Slide kernel across the image using center indexing
    for i in range(center_h, img_h - center_h):
        for j in range(center_w, img_w - center_w):
            # Extract patch centered at current position
            patch = image[
                i - center_h : i + center_h + 1, j - center_w : j + center_w + 1
            ]
            yield (i, j), patch.astype(np.float32), kernel.astype(np.float32)


def conv2d_center_indexed(image, kernel):
    # Get dimensions
    output = np.zeros(image.shape[:2])
    for (i, j), patch, kernel in get_patch_and_kernel_center(image, kernel):
        output[i, j] = single_conv_operation(patch, kernel)
    return output

In [ ]:
p = Path(
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/1038155225184409"
)
img, mask = DiskImage.load(p / "image.jpg"), DiskBooleanMask.load(p / "mask.png")

crop = img[350:450, -135:-35]
plt.imshow(crop)

In [ ]:
INPUT_IMG_DIR = Path(
    "/Users/hariomnarang/Desktop/personal/roads/datasets/interpretation/smallnet/inputs/"
)

In [ ]:
! ls "/Users/hariomnarang/Desktop/personal/roads/datasets/interpretation/smallnet/inputs/"

In [ ]:
jalebi = plt.imread(INPUT_IMG_DIR / "jalebi"/ "image.jpg")
gray_jalebi = cv2.cvtColor(jalebi, cv2.COLOR_RGB2GRAY)

In [ ]:
from mtrain.utils import show_with_custom_limit

sobel = np.array(
    [
        [-1, 0, 1],
        [-2, 0, 2],
        [-1, 0, 1],
    ]
)

sobel_h = np.array(
    [
        [-1, -2, -1],
        [0, 0, 0],
        [1, 2, 1],
    ]
)

gray_crop = cv2.cvtColor(crop, cv2.COLOR_RGB2GRAY)
show(
    [
        gray_crop,
        conv2d_center_indexed(gray_crop, sobel),
        conv2d_center_indexed(gray_crop, sobel_h),
        conv2d_center_indexed(gray_crop, sobel) + conv2d_center_indexed(gray_crop, sobel_h),
    ],
    (20,20),
    cmap="gray",
    ncols=4,
)

In [ ]:
plt.imshow(
    conv2d_pytorch_style(gray_jalebi, sobel) + conv2d_pytorch_style(gray_jalebi, sobel_h), 
cmap="gray")

In [ ]:
from mtrain.interp.analysis import list_layers, get_layer_data, to_weight_id, to_bias_id
def get_weights_and_acts(weights_dir, activations_dir):
    activation_layer_ids = list_layers(activations_dir)
    weights_layer_ids = set(list_layers(weights_dir))
    for layer_id in activation_layer_ids:
        w_layer_id = to_weight_id(layer_id)
        b_layer_id = to_bias_id(layer_id)

        activation = get_layer_data(activations_dir, layer_id)
        weight, bias = None, None
        if b_layer_id in weights_layer_ids:
            bias = get_layer_data(weights_dir, b_layer_id)
        if w_layer_id in weights_layer_ids:
            weight = get_layer_data(weights_dir, w_layer_id)
        yield {"activation": activation, "weight": weight, "bias": bias}

In [ ]:
WEIGHTS_DIR = Path("/Users/hariomnarang/Desktop/personal/roads/datasets/interpretation/smallnet/weights")
ACTIVATIONS_DIR = Path("/Users/hariomnarang/Desktop/personal/roads/datasets/interpretation/smallnet/analysis/solid_orange")
iterator = get_weights_and_acts(WEIGHTS_DIR, ACTIVATIONS_DIR)

In [ ]:
act = next(iterator)

In [ ]:
show_single_channel_red_green_black(act["activation"]["data"][0], ncols=8)

In [ ]:
show_single_channel_red_green_black(
    [patch, kernel, patch * kernel],
    ncols=3,
    limit_getter_type="local_same",
)

# Images dataset

In [ ]:
from pathlib import Path
SAVE_DIR = Path("/Users/hariomnarang/Desktop/personal/roads/datasets/interpretation/smallnet/inputs")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def solid_color(shape, color_rgb):
    return np.ones((shape[0], shape[1], 3), dtype=np.uint8) * np.array(color_rgb, dtype=np.uint8)

plt.imsave(
    mkdir(SAVE_DIR / "solid_orange") / "image.jpg",
    solid_color([100,100], [255,165,0])
)